# Apache Iceberg com PySpark
## Cenário

Loja virtual fictícia com duas tabelas: **clientes** e **pedidos** (1 cliente : N pedidos).
O conjunto de dados foi **criado pelo grupo** (`data/clientes.csv` e `data/pedidos.csv`).

![Modelo ER](../../assets/modelo_er.svg)

In [1]:
import sys; sys.path.append("../..")
from pesquisadados.main import spark_iceberg as criar_spark, DADOS
spark = criar_spark()
spark.version

26/10/03 21:37:02 WARN Utils: Your hostname, DESKTOP-73OKR5D resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/03 21:37:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thomasmaggi/pesquisadados/pesquisadados/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thomasmaggi/.ivy2/cache
The jars for the packages stored in: /home/thomasmaggi/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8a3b1e03-2044-4cbf-a305-22b7b6bfab04;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
:: resolution report :: resolve 152ms :: artifacts dl 3ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org

'3.5.3'

## DDL

In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.loja")
spark.sql("DROP TABLE IF EXISTS local.loja.clientes PURGE")
spark.sql("DROP TABLE IF EXISTS local.loja.pedidos PURGE")
spark.sql("""
CREATE TABLE local.loja.clientes (
  id_cliente INT, nome STRING, cidade STRING, uf STRING
) USING iceberg 
""")
spark.sql("""
CREATE TABLE local.loja.pedidos (
  id_pedido INT, id_cliente INT, produto STRING, quantidade INT,
  valor_unitario DECIMAL(10,2), data_pedido DATE, status STRING
) USING iceberg 
""")

DataFrame[]

## Carga inicial (INSERT a partir do CSV)

In [3]:
clientes = spark.read.csv(str(DADOS/"clientes.csv"), header=True, inferSchema=True)
pedidos = spark.read.csv(str(DADOS/"pedidos.csv"), header=True, inferSchema=True)
clientes.createOrReplaceTempView("v_clientes"); pedidos.createOrReplaceTempView("v_pedidos")
spark.sql("INSERT INTO local.loja.clientes SELECT id_cliente, nome, cidade, uf FROM v_clientes")
spark.sql("""INSERT INTO local.loja.pedidos
SELECT id_pedido, id_cliente, produto, quantidade, CAST(valor_unitario AS DECIMAL(10,2)),
       CAST(data_pedido AS DATE), status FROM v_pedidos""")
spark.table("local.loja.pedidos").show()

+---------+----------+--------+----------+--------------+-----------+---------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+--------+----------+--------------+-----------+---------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|     PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|     PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03| PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05| PENDENTE|
|      105|         4| Headset|         1|        300.00| 2026-09-06|CANCELADO|
+---------+----------+--------+----------+--------------+-----------+---------+



## INSERT
Insere um novo pedido.

In [4]:
spark.sql("""INSERT INTO local.loja.pedidos VALUES
(106, 2, 'Webcam', 1, 350.00, DATE'2026-09-10', 'PENDENTE')""")
spark.table("local.loja.pedidos").orderBy("id_pedido").show()

+---------+----------+--------+----------+--------------+-----------+---------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+--------+----------+--------------+-----------+---------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|     PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|     PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03| PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05| PENDENTE|
|      105|         4| Headset|         1|        300.00| 2026-09-06|CANCELADO|
|      106|         2|  Webcam|         1|        350.00| 2026-09-10| PENDENTE|
+---------+----------+--------+----------+--------------+-----------+---------+



## UPDATE
Marca como PAGO os pedidos pendentes do cliente 1.

In [5]:
spark.sql("UPDATE local.loja.pedidos SET status = 'PAGO' WHERE id_cliente = 1 AND status = 'PENDENTE'")
spark.table("local.loja.pedidos").orderBy("id_pedido").show()

+---------+----------+--------+----------+--------------+-----------+---------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|   status|
+---------+----------+--------+----------+--------------+-----------+---------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|     PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|     PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03| PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05|     PAGO|
|      105|         4| Headset|         1|        300.00| 2026-09-06|CANCELADO|
|      106|         2|  Webcam|         1|        350.00| 2026-09-10| PENDENTE|
+---------+----------+--------+----------+--------------+-----------+---------+



## DELETE
Remove os pedidos cancelados.

In [6]:
spark.sql("DELETE FROM local.loja.pedidos WHERE status = 'CANCELADO'")
spark.table("local.loja.pedidos").orderBy("id_pedido").show()

+---------+----------+--------+----------+--------------+-----------+--------+
|id_pedido|id_cliente| produto|quantidade|valor_unitario|data_pedido|  status|
+---------+----------+--------+----------+--------------+-----------+--------+
|      101|         1|Notebook|         1|       4500.00| 2026-09-01|    PAGO|
|      102|         2|   Mouse|         2|         80.00| 2026-09-02|    PAGO|
|      103|         3| Monitor|         1|       1200.00| 2026-09-03|PENDENTE|
|      104|         1| Teclado|         1|        250.00| 2026-09-05|    PAGO|
|      106|         2|  Webcam|         1|        350.00| 2026-09-10|PENDENTE|
+---------+----------+--------+----------+--------------+-----------+--------+



## Histórico de versões

In [7]:
spark.sql("SELECT snapshot_id, operation FROM local.loja.pedidos.snapshots").show(truncate=False)

+-------------------+---------+
|snapshot_id        |operation|
+-------------------+---------+
|5614931194731159847|append   |
|7729203565259862681|append   |
|7243719340899915155|overwrite|
|6089313181295806824|overwrite|
+-------------------+---------+



In [8]:
spark.stop()